In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from PIL import Image, ImageEnhance, ImageFilter

# --- Preprocessing Functions ---
def apply_pst(image, alpha=0.1, beta=0.1):
    """Apply Phase Stretch Transform (PST) to the image."""
    image = np.array(image, dtype=np.float32) / 255.0
    fft_image = np.fft.fft2(image)
    magnitude = np.abs(fft_image)
    phase = np.angle(fft_image)
    phase_stretched = phase + alpha * np.tanh(beta * phase)
    pst_image = np.abs(np.fft.ifft2(magnitude * np.exp(1j * phase_stretched)))
    return pst_image

def apply_qhed(image):
    """Apply Quantile Histogram Equalization and Denoising (QHED) to the image."""
    image = image.convert("RGB")
    enhancer = ImageEnhance.Contrast(image)
    enhanced_image = enhancer.enhance(2.0)
    denoised_image = enhanced_image.filter(ImageFilter.MedianFilter(size=3))
    return denoised_image

# --- Dataset Class ---
class TripleInputDataset(datasets.ImageFolder):
    """Dataset that returns raw, PST, and QHED images."""
    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")

        # Raw Image
        raw_image = self.transform(image)

        # PST Preprocessed Image
        pst_image = apply_pst(image)
        pst_image = Image.fromarray((pst_image * 255).astype(np.uint8)).convert("RGB")
        pst_image = self.transform(pst_image)

        # QHED Preprocessed Image
        qhed_image = apply_qhed(image)
        qhed_image = self.transform(qhed_image)

        return raw_image, pst_image, qhed_image, label

# --- Residual Block ---
class ResidualBlock(nn.Module):
    """A residual block with convolutional layers and skip connections."""
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.skip(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        return self.relu(out)

# --- ECNN High-Capacity Model ---
class ECNNHighCapacity(nn.Module):
    """High-Capacity Ensemble CNN with Residual Blocks."""
    def __init__(self, num_classes):
        super(ECNNHighCapacity, self).__init__()
        self.raw_cnn = self._create_cnn_branch()
        self.pst_cnn = self._create_cnn_branch()
        self.qhed_cnn = self._create_cnn_branch()

        # Combine features from all three branches and reduce to 512
        self.fc = nn.Sequential(
            nn.Linear(256 * 3, 512),  # Combine and reduce features to 512
            nn.ReLU(),
            nn.Dropout(0.5),
        )

    def _create_cnn_branch(self):
        return nn.Sequential(
            ResidualBlock(3, 64),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128),
            ResidualBlock(128, 256, stride=2),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, raw_x, pst_x, qhed_x):
        # Process each input branch
        raw_features = self.raw_cnn(raw_x).view(raw_x.size(0), -1)  # Shape: (batch_size, 256)
        pst_features = self.pst_cnn(pst_x).view(pst_x.size(0), -1)  # Shape: (batch_size, 256)
        qhed_features = self.qhed_cnn(qhed_x).view(qhed_x.size(0), -1)  # Shape: (batch_size, 256)

        # Concatenate features
        combined_features = torch.cat((raw_features, pst_features, qhed_features), dim=1)  # Shape: (batch_size, 768)

        # Reduce to 512
        return self.fc(combined_features)  # Shape: (batch_size, 512)


# --- Combined Fine-Tuned Model ---
class CombinedFineTunedModel(nn.Module):
    """Fine-tuned model combining ViT and ECNN outputs."""
    def __init__(self, num_classes):
        super(CombinedFineTunedModel, self).__init__()
        # Pretrained ViT with hidden states enabled
        self.vit = ViTForImageClassification.from_pretrained(
            "google/vit-base-patch16-224-in21k",
            output_hidden_states=True  # Enable hidden states for feature extraction
        )
        self.vit_hidden_size = 768  # Default hidden size of ViT
        self.ecnn = ECNNHighCapacity(num_classes)  # ECNN with raw, PST, and QHED inputs

        # Linear projection to align ViT features with ECNN features
        self.vit_projection = nn.Linear(self.vit_hidden_size, 512)

        # Fully connected layer to combine ViT and ECNN features
        self.fc = nn.Sequential(
            nn.Linear(512 + 512, 256),  # Combine ViT (512) + ECNN (512)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)  # Final classification layer
        )

    def forward(self, raw_x, pst_x, qhed_x):
        # Extract ViT features from the CLS token
        vit_outputs = self.vit(raw_x, output_hidden_states=True)
        vit_hidden_states = vit_outputs.hidden_states[-1]  # Last hidden state
        vit_cls_token = vit_hidden_states[:, 0, :]  # CLS token (batch_size, 768)
        vit_features = self.vit_projection(vit_cls_token)  # Project to (batch_size, 512)

        # Get ECNN features
        ecnn_features = self.ecnn(raw_x, pst_x, qhed_x)  # Output: (batch_size, 512)

        # Concatenate features from ViT and ECNN
        combined_features = torch.cat((vit_features, ecnn_features), dim=1)  # Shape: (batch_size, 1024)

        # Pass through the final fully connected layer
        return self.fc(combined_features)


# --- Training Setup ---
train_dir = "/kaggle/input/binary-augmented-split-adhd/ADHD - Augmented Split Binary/train"
val_dir = "/kaggle/input/binary-augmented-split-adhd/ADHD - Augmented Split Binary/validation"

# Dataset Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Datasets and Loaders
batch_size = 8
train_dataset = TripleInputDataset(root=train_dir, transform=transform)
val_dataset = TripleInputDataset(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Instantiate Model
num_classes = 2
model = CombinedFineTunedModel(num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer, Scheduler, and Loss
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

# --- Save Checkpoint ---
def save_checkpoint(epoch, model, optimizer, scheduler, train_loss, val_loss, train_acc, val_acc, path):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved: {path}")

# --- Training Loop ---
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for raw_x, pst_x, qhed_x, labels in train_loader:
        raw_x, pst_x, qhed_x, labels = raw_x.to(device), pst_x.to(device), qhed_x.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(raw_x, pst_x, qhed_x)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")

    # Validation Loop
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for raw_x, pst_x, qhed_x, labels in val_loader:
            raw_x, pst_x, qhed_x, labels = raw_x.to(device), pst_x.to(device), qhed_x.to(device), labels.to(device)
            outputs = model(raw_x, pst_x, qhed_x)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Save Model
    checkpoint_path = f"fine_tuned_combined_model_epoch_{epoch+1}.pth"
    save_checkpoint(
        epoch=epoch + 1,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loss=train_loss,
        val_loss=val_loss,
        train_acc=train_acc,
        val_acc=val_acc,
        path=checkpoint_path,
    )
